# mT0-small Gold-context Seq2Seq RAG

Here is used **`bigscience/mt0-small`** instruction-tuned Seq2Seq generator.

`processed_question + gold context -> processed_answer`

After training the same generator is used in 3 RAG configs:

1. **TF-IDF + mT0-small**
2. **BM25 + mT0-small**
3. **Dense + mT0-small**

Validation set is used only for `eval_loss`, early stopping and picking best checkpointa.  


Experiment Rationale

The initial Seq2Seq generator was based on `google/mt5-base` and was fine-tuned on a small Serbian question-answering dataset using gold context. Although the validation loss decreased during training, the generated answers revealed that the model had not learned the QA task reliably.

A recurring issue was the generation of T5 sentinel tokens such as `<extra_id_0>`. These tokens originate from the span-corruption pretraining objective used by T5/mT5 models. Their persistent appearance after fine-tuning indicated that the model was still strongly influenced by its pretraining behavior instead of consistently producing direct answers.

Several adjustments were tested, including FP32 training, a higher learning rate, more training epochs, improved prompting, gradient clipping, and explicit suppression of sentinel tokens during generation. These changes improved the validation loss and produced more domain-related text, but the model still failed to answer both validation and even training examples reliably. Generated outputs were often only loosely related to the question, repetitive, or extracted from an incorrect part of the context.

Because the training set contains only around one hundred QA examples, continuing to fine-tune a non-instruction-tuned `mT5-base` model was considered unlikely to provide sufficient improvement within the project constraints.

Therefore, the generator was replaced with `bigscience/mt0-small`. mT0 is still a multilingual encoder-decoder Seq2Seq model based on the T5/mT5 architecture, but it has additionally been instruction-tuned. This makes it a more suitable starting point for a question-answering task with limited supervised training data.

The overall experimental design remains unchanged: the generator is trained using gold context and is later evaluated in three RAG configurations that differ only in the retrieval component: TF-IDF, BM25, and dense retrieval.


In [19]:
# # Kloniranje repozitorijuma samo ako već ne postoji.
# from pathlib import Path

# REPO_PATH = Path("/content/Student-Question-Answering-from-Course-Materials")

# if not REPO_PATH.exists():
#     !git clone https://github.com/anjaanjaa10/Student-Question-Answering-from-Course-Materials.git

# %cd /content/Student-Question-Answering-from-Course-Materials


/content/Student-Question-Answering-from-Course-Materials


In [20]:
# from google.colab import drive

# drive.mount("/content/drive")
# #

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import json
import math
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)


In [23]:
SEED = 42

EXPERIMENT_NAME = "mt0_small_gold_v2"
MODEL_NAME = "bigscience/mt0-small"

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

RAG_TOP_K = 5
MAX_GOLD_CONTEXT_CHUNKS = RAG_TOP_K

LEARNING_RATE = 3e-4
NUM_TRAIN_EPOCHS = 12
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.10
EARLY_STOPPING_PATIENCE = 3
MAX_GRAD_NORM = 1.0

GENERATION_MAX_NEW_TOKENS = 128
GENERATION_NUM_BEAMS = 4
GENERATION_BATCH_SIZE = 1

TOP_K_EXPORTED = 10

USE_FP16 = False
USE_BF16 = False

set_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


DEVICE: cuda
CUDA: True
GPU: Tesla T4


In [24]:
PROJECT_ROOT = Path(
    "/content/Student-Question-Answering-from-Course-Materials"
)

if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError(
        "Nije pronađen folder 'data' u root direktorijumu projekta."
    )

CHUNKS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks_preprocessed.jsonl"
)

TRAIN_PATH = PROJECT_ROOT / "data" / "splits" / "train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data" / "splits" / "validation.jsonl"
TEST_PATH = PROJECT_ROOT / "data" / "splits" / "test.jsonl"

RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"

ARTIFACTS_DIR = Path(
    "/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2"
)

CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints"
BEST_MODEL_DIR = ARTIFACTS_DIR / "best_model"
PREPARED_DATA_DIR = ARTIFACTS_DIR / "prepared_data"
GENERATION_DIR = ARTIFACTS_DIR / "generation"
RAG_CONFIG_DIR = ARTIFACTS_DIR / "rag_configs"

for directory in [
    ARTIFACTS_DIR,
    CHECKPOINT_DIR,
    BEST_MODEL_DIR,
    PREPARED_DATA_DIR,
    GENERATION_DIR,
    RAG_CONFIG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACTS_DIR / "training.log"

logger = logging.getLogger("mt0_small_gold")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(
    LOG_PATH,
    mode="a",
    encoding="utf-8"
)
file_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )
)
logger.addHandler(file_handler)

logger.info("Notebook started.")
logger.info("Device: %s", DEVICE)
logger.info("Model: %s", MODEL_NAME)


INFO:mt0_small_gold:Notebook started.
INFO:mt0_small_gold:Device: cuda
INFO:mt0_small_gold:Model: bigscience/mt0-small


In [25]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def save_jsonl(records: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                )
                + "\n"
            )


for required_path in [
    CHUNKS_PATH,
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)


chunks = load_jsonl(CHUNKS_PATH)
train_data = load_jsonl(TRAIN_PATH)
validation_data = load_jsonl(VALIDATION_PATH)
test_data = load_jsonl(TEST_PATH)

print("Chunks:", len(chunks))
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))


Chunks: 356
Train: 100
Validation: 21
Test: 22


## Gold context

During training retriever is not used.
Context is chosen only by annotated `source_pages`, so generator is trained on gold.


In [26]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

TASK_PREFIX = (
    "Na osnovu konteksta odgovori na pitanje na srpskom jeziku."
)


In [27]:
def chunk_pages(chunk: dict) -> set[int]:
    return set(
        range(
            int(chunk["pdf_page_start"]),
            int(chunk["pdf_page_end"]) + 1
        )
    )


def get_gold_chunks(
    source_pages,
    max_chunks: int = MAX_GOLD_CONTEXT_CHUNKS
) -> list[dict]:

    if isinstance(source_pages, int):
        source_pages = [source_pages]

    source_pages = set(source_pages)
    candidates = []

    for corpus_index, chunk in enumerate(chunks):
        overlap = len(
            chunk_pages(chunk) & source_pages
        )

        if overlap > 0:
            candidates.append({
                "overlap": overlap,
                "corpus_index": corpus_index,
                "chunk": chunk,
            })

    candidates.sort(
        key=lambda item: (
            -item["overlap"],
            item["corpus_index"]
        )
    )

    selected = candidates[:max_chunks]

    selected.sort(
        key=lambda item: item["corpus_index"]
    )

    return [
        item["chunk"]
        for item in selected
    ]


def build_model_input(
    processed_question: str,
    context: str
) -> str:
    return (
        f"{TASK_PREFIX}\n"
        f"Pitanje: {processed_question}\n"
        f"Kontekst:\n{context}\n"
        f"Odgovor:"
    )


def token_length(text: str) -> int:
    return len(
        tokenizer(
            text,
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )


def pack_context_to_input_budget(
    processed_question: str,
    selected_chunks: list[dict],
    max_input_length: int = MAX_INPUT_LENGTH,
) -> tuple[str, int, bool]:

    separator = "\n\n---\n\n"
    accepted_parts = []
    represented_chunks = 0
    shortened_last_chunk = False

    for chunk in selected_chunks:
        text = chunk.get("processed_text", "").strip()

        if not text:
            continue

        candidate_parts = accepted_parts + [text]
        candidate_context = separator.join(candidate_parts)

        if token_length(
            build_model_input(
                processed_question,
                candidate_context
            )
        ) <= max_input_length:
            accepted_parts.append(text)
            represented_chunks += 1
            continue

        chunk_token_ids = tokenizer(
            text,
            truncation=False,
            add_special_tokens=False
        )["input_ids"]

        low = 0
        high = len(chunk_token_ids)
        best_prefix = ""

        while low <= high:
            mid = (low + high) // 2

            prefix = tokenizer.decode(
                chunk_token_ids[:mid],
                skip_special_tokens=True
            ).strip()

            partial_parts = (
                accepted_parts + [prefix]
                if prefix
                else accepted_parts
            )

            partial_context = separator.join(
                partial_parts
            )

            current_length = token_length(
                build_model_input(
                    processed_question,
                    partial_context
                )
            )

            if current_length <= max_input_length:
                best_prefix = prefix
                low = mid + 1
            else:
                high = mid - 1

        if best_prefix:
            accepted_parts.append(best_prefix)
            represented_chunks += 1
            shortened_last_chunk = True

        break

    context = separator.join(accepted_parts)

    return (
        context,
        represented_chunks,
        shortened_last_chunk
    )


In [28]:
def prepare_gold_examples(
    data: list[dict],
    split_name: str
) -> list[dict]:

    prepared = []
    missing_context_ids = []

    for example in data:
        gold_chunks = get_gold_chunks(
            example["source_pages"]
        )

        (
            context,
            n_context_chunks_used,
            context_shortened,
        ) = pack_context_to_input_budget(
            example["processed_question"],
            gold_chunks,
        )

        if not context.strip():
            missing_context_ids.append(
                example["id"]
            )
            continue

        target = example.get(
            "processed_answer",
            example.get("answer", "")
        ).strip()

        if not target:
            raise ValueError(
                f"Nedostaje target odgovor za ID "
                f"{example['id']} u splitu {split_name}."
            )

        model_input = build_model_input(
            example["processed_question"],
            context
        )

        if token_length(model_input) > MAX_INPUT_LENGTH:
            raise RuntimeError(
                f"Input za ID {example['id']} je duži od "
                f"{MAX_INPUT_LENGTH} tokena."
            )

        prepared.append({
            "question_id": example["id"],
            "input_text": model_input,
            "target_text": target,
            "source_pages": example["source_pages"],
            "n_gold_chunks_available": len(gold_chunks),
            "n_context_chunks_used": n_context_chunks_used,
            "context_shortened": context_shortened,
        })

    if missing_context_ids:
        raise ValueError(
            f"Gold context nedostaje za {split_name} IDs: "
            f"{missing_context_ids}"
        )

    logger.info(
        "%s prepared examples: %d",
        split_name,
        len(prepared)
    )

    return prepared


train_gold = prepare_gold_examples(
    train_data,
    "train"
)

validation_gold = prepare_gold_examples(
    validation_data,
    "validation"
)

save_jsonl(
    train_gold,
    PREPARED_DATA_DIR / "train_gold.jsonl"
)

save_jsonl(
    validation_gold,
    PREPARED_DATA_DIR / "validation_gold.jsonl"
)

print("Prepared train:", len(train_gold))
print("Prepared validation:", len(validation_gold))


INFO:mt0_small_gold:train prepared examples: 100
INFO:mt0_small_gold:validation prepared examples: 21


Prepared train: 100
Prepared validation: 21


In [29]:
def sequence_lengths(
    records: list[dict],
    field: str
) -> list[int]:
    return [
        token_length(record[field])
        for record in records
    ]


train_input_lengths = sequence_lengths(
    train_gold,
    "input_text"
)

validation_input_lengths = sequence_lengths(
    validation_gold,
    "input_text"
)

train_target_lengths = sequence_lengths(
    train_gold,
    "target_text"
)

validation_target_lengths = sequence_lengths(
    validation_gold,
    "target_text"
)

length_stats = {
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "train": {
        "n": len(train_gold),
        "input_mean": float(np.mean(train_input_lengths)),
        "input_max": int(max(train_input_lengths)),
        "target_max": int(max(train_target_lengths)),
        "contexts_shortened": int(
            sum(
                bool(record["context_shortened"])
                for record in train_gold
            )
        ),
    },
    "validation": {
        "n": len(validation_gold),
        "input_mean": float(np.mean(validation_input_lengths)),
        "input_max": int(max(validation_input_lengths)),
        "target_max": int(max(validation_target_lengths)),
        "contexts_shortened": int(
            sum(
                bool(record["context_shortened"])
                for record in validation_gold
            )
        ),
    },
}

with (ARTIFACTS_DIR / "input_length_stats.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        length_stats,
        file,
        ensure_ascii=False,
        indent=2
    )

print(json.dumps(
    length_stats,
    ensure_ascii=False,
    indent=2
))


{
  "max_input_length": 512,
  "max_target_length": 128,
  "train": {
    "n": 100,
    "input_mean": 511.9,
    "input_max": 512,
    "target_max": 138,
    "contexts_shortened": 100
  },
  "validation": {
    "n": 21,
    "input_mean": 512.0,
    "input_max": 512,
    "target_max": 119,
    "contexts_shortened": 21
  }
}


## Tokenization

Target contains only reference abswer; we are not adding T5 sentinel tokens.


In [30]:
train_dataset = Dataset.from_list(
    train_gold
)

validation_dataset = Dataset.from_list(
    validation_gold
)


def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_validation = validation_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=validation_dataset.column_names
)


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

In [31]:
# Sanity check targeta pre treninga
for i in range(min(3, len(tokenized_train))):
    print("=" * 100)
    print("RAW TARGET:")
    print(train_gold[i]["target_text"])

    print("\nTOKENIZED TARGET:")
    print(
        tokenizer.decode(
            tokenized_train[i]["labels"],
            skip_special_tokens=False
        )
    )


RAW TARGET:
Primeri neprijatnosti su iznenadno gašenje internet pregledača, muzičkog plejera ili mobilne igre, kao i greška u sistemu za izgradnju softvera. Ozbiljnije neprijatnosti mogu nastati kada navigacioni softver pogrešno usmeri korisnika ili kada softverski problem onemogući uspostavljanje poziva u hitnoj situaciji. Primer materijalnog gubitka je softverska greška u kompaniji Knight Capital Group, zbog koje je nastao gubitak od 440 miliona dolara za 45 minuta.

TOKENIZED TARGET:
Primeri neprijatnosti su iznenadno gašenje internet pregledača, muzičkog plejera ili mobilne igre, kao i greška u sistemu za izgradnju softvera. Ozbiljnije neprijatnosti mogu nastati kada navigacioni softver pogrešno usmeri korisnika ili kada softverski problem onemogući uspostavljanje poziva u hitnoj situaciji. Primer materijalnog gubitka je softverska greška u kompaniji Knight Capital Group, zbog koje je nastao gubitak od</s>
RAW TARGET:
Kod sistema za kupovinu avionskih karata mogu se zadati zahtevi 

## Zero-shot check before fine-tuninga

We see behaviour of instruction-tuned `mT0-small` model before we train it


In [32]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(DEVICE)

model.eval()

for i in range(min(3, len(validation_gold))):
    example = validation_gold[i]

    encoded = tokenizer(
        example["input_text"],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **encoded,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            num_beams=GENERATION_NUM_BEAMS,
        )

    prediction = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    print("=" * 100)
    print(f"ZERO-SHOT PRIMER {i + 1}")
    print("\nREFERENTNI ODGOVOR:")
    print(example["target_text"])
    print("\nGENERISANI ODGOVOR:")
    print(prediction)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


ZERO-SHOT PRIMER 1

REFERENTNI ODGOVOR:
Cachegrind je Valgrind alat za profilisanje keš memorije. Koristi se pokretanjem programa kroz Cachegrind, nakon čega se analiziraju prikupljene informacije o pristupima kešu i izvršavanju.

GENERISANI ODGOVOR:
brojač
ZERO-SHOT PRIMER 2

REFERENTNI ODGOVOR:
Instrumentaciono profajliranje koristi dodatno ubačen kod za prikupljanje tačnih podataka o izvršavanju, na primer broja poziva funkcija ili izvršenih grana i blokova.

GENERISANI ODGOVOR:
6.2: Deo plamenog grafa za izvršavanje program Game of Life u Javi Širina
ZERO-SHOT PRIMER 3

REFERENTNI ODGOVOR:
Dobar skup testova treba da efikasno otkriva greške, bude relativno mali i brz za izvršavanje i da pruža visok stepen poverenja u pouzdanost softvera.

GENERISANI ODGOVOR:
ijama za testiranje tj. kolekcijama uređaja i računara sa različitam softverskim konfiguracijama


## Fine-tuning kconfiguration

Trening is in FP32 for stability stabilnosti. Best checkpoint is chosen by validation `eval_loss`


In [33]:
# Tokom treninga cache se isključuje zbog gradient checkpointing-a.
model.config.use_cache = False

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

optimizer_steps_per_epoch = math.ceil(
    len(tokenized_train)
    / (
        TRAIN_BATCH_SIZE
        * GRADIENT_ACCUMULATION_STEPS
    )
)

TOTAL_OPTIMIZER_STEPS = (
    optimizer_steps_per_epoch
    * NUM_TRAIN_EPOCHS
)

WARMUP_STEPS = max(
    1,
    round(
        WARMUP_FRACTION
        * TOTAL_OPTIMIZER_STEPS
    )
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    max_grad_norm=MAX_GRAD_NORM,

    num_train_epochs=NUM_TRAIN_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    gradient_checkpointing=True,

    optim="adafactor",
    torch_empty_cache_steps=1,

    bf16=USE_BF16,
    fp16=USE_FP16,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    predict_with_generate=False,

    report_to="none",
    disable_tqdm=True,

    seed=SEED,
    data_seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ],
)

print(
    "Optimizer steps/epoch:",
    optimizer_steps_per_epoch
)
print(
    "Total optimizer steps:",
    TOTAL_OPTIMIZER_STEPS
)
print(
    "Warmup steps:",
    WARMUP_STEPS
)


Optimizer steps/epoch: 25
Total optimizer steps: 300
Warmup steps: 30


## Training

Only cell to run real fine-tuning


In [34]:
train_result = trainer.train()

print("BEST CHECKPOINT:")
print(trainer.state.best_model_checkpoint)

print("\nBEST METRIC:")
print(trainer.state.best_metric)

trainer.save_model(
    str(BEST_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(BEST_MODEL_DIR)
)

training_history_df = pd.DataFrame(
    trainer.state.log_history
)

training_history_df.to_csv(
    ARTIFACTS_DIR / "training_history.csv",
    index=False
)

train_metrics = {
    key: (
        float(value)
        if isinstance(value, (int, float, np.number))
        else value
    )
    for key, value in train_result.metrics.items()
}

with (ARTIFACTS_DIR / "train_metrics.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        train_metrics,
        file,
        ensure_ascii=False,
        indent=2
    )

final_eval_metrics = trainer.evaluate()

with (ARTIFACTS_DIR / "validation_loss_metrics.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            key: float(value)
            if isinstance(value, (int, float, np.number))
            else value
            for key, value in final_eval_metrics.items()
        },
        file,
        ensure_ascii=False,
        indent=2
    )


{'loss': '23.29', 'grad_norm': '22.52', 'learning_rate': '0.00024', 'epoch': '1'}
{'eval_loss': '5.068', 'eval_runtime': '1.259', 'eval_samples_per_second': '16.68', 'eval_steps_per_second': '16.68', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '19.47', 'grad_norm': '26.19', 'learning_rate': '0.0002789', 'epoch': '2'}
{'eval_loss': '4.341', 'eval_runtime': '1.346', 'eval_samples_per_second': '15.6', 'eval_steps_per_second': '15.6', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '17.01', 'grad_norm': '15.55', 'learning_rate': '0.0002511', 'epoch': '3'}
{'eval_loss': '4.069', 'eval_runtime': '0.725', 'eval_samples_per_second': '28.96', 'eval_steps_per_second': '28.96', 'epoch': '3'}


...

{'train_runtime': '524.4', 'train_samples_per_second': '2.288', 'train_steps_per_second': '0.572', 'train_loss': '14.92', 'epoch': '10'}


[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


BEST CHECKPOINT:
/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/checkpoints/checkpoint-175

BEST METRIC:
3.790537118911743


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '3.791', 'eval_runtime': '1.161', 'eval_samples_per_second': '18.1', 'eval_steps_per_second': '18.1', 'epoch': '10'}


In [35]:
experiment_config = {
    "experiment": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "training_context": "gold_token_budgeted",
    "generator_document_field": "processed_text",
    "generator_question_field": "processed_question",
    "target_field": "processed_answer",
    "seed": SEED,
    "max_input_length": MAX_INPUT_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "rag_top_k": RAG_TOP_K,
    "max_gold_context_chunks": MAX_GOLD_CONTEXT_CHUNKS,
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "optimizer": "adafactor",
    "fp16": USE_FP16,
    "bf16": USE_BF16,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": trainer.state.best_metric,
    "n_train": len(train_gold),
    "n_validation": len(validation_gold),
}

with (ARTIFACTS_DIR / "experiment_config.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_config,
        file,
        ensure_ascii=False,
        indent=2
    )


## Gold-context check of the best model

Before RAG inference check if generator gives okay answer when it gets apropriate gold context


In [36]:
model_for_check = trainer.model
model_for_check.eval()

gold_validation_predictions = []

for i in range(min(5, len(validation_gold))):
    example = validation_gold[i]

    encoded = tokenizer(
        example["input_text"],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model_for_check.generate(
            **encoded,
            max_new_tokens=GENERATION_MAX_NEW_TOKENS,
            num_beams=GENERATION_NUM_BEAMS,
            no_repeat_ngram_size=3,
        )

    prediction = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
    ).strip()

    gold_validation_predictions.append({
        "question_id": example["question_id"],
        "gold_answer": example["target_text"],
        "generated_answer": prediction,
    })

    print("=" * 100)
    print(f"PRIMER {i + 1}")
    print("\nREFERENTNI ODGOVOR:")
    print(example["target_text"])
    print("\nGENERISANI ODGOVOR:")
    print(prediction)

save_jsonl(
    gold_validation_predictions,
    ARTIFACTS_DIR / "gold_validation_sample_predictions.jsonl"
)


PRIMER 1

REFERENTNI ODGOVOR:
Cachegrind je Valgrind alat za profilisanje keš memorije. Koristi se pokretanjem programa kroz Cachegrind, nakon čega se analiziraju prikupljene informacije o pristupima kešu i izvršavanju.

GENERISANI ODGOVOR:
Cachegrind je za šta služi i kako se koristi. Obuhvata vreme izvršavanja ili povećava potrebnu memoriju kao i vreme kompilacije.
PRIMER 2

REFERENTNI ODGOVOR:
Instrumentaciono profajliranje koristi dodatno ubačen kod za prikupljanje tačnih podataka o izvršavanju, na primer broja poziva funkcija ili izvršenih grana i blokova.

GENERISANI ODGOVOR:
Instrumentaciono profajliranje je obično interaktivan prikaz poziva metoda tokom izvršavanja programa.
PRIMER 3

REFERENTNI ODGOVOR:
Dobar skup testova treba da efikasno otkriva greške, bude relativno mali i brz za izvršavanje i da pruža visok stepen poverenja u pouzdanost softvera.

GENERISANI ODGOVOR:
Komponenti su karakteristike dobrog skupa testova, kako bi se osiguralo da instalacija teče bez grešaka i 

# RAG inference

From now on generator is fixed. Only retrieval result is changing.


In [37]:
inference_tokenizer = AutoTokenizer.from_pretrained(
    BEST_MODEL_DIR
)

inference_model = AutoModelForSeq2SeqLM.from_pretrained(
    BEST_MODEL_DIR
).to(DEVICE)

inference_model.config.use_cache = True
inference_model.eval()

print("Inference model loaded from:")
print(BEST_MODEL_DIR)


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Inference model loaded from:
/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/best_model


In [38]:
def build_rag_input(
    retrieval_record: dict,
    top_k: int = RAG_TOP_K
) -> str:

    retrieved_chunks = retrieval_record.get(
        "retrieved_chunks",
        []
    )[:top_k]

    question = retrieval_record.get(
        "processed_question",
        retrieval_record.get("question", "")
    )

    context, _, _ = pack_context_to_input_budget(
        question,
        retrieved_chunks,
        max_input_length=MAX_INPUT_LENGTH
    )

    return build_model_input(
        question,
        context
    )


In [39]:
@torch.inference_mode()
def generate_batch(
    input_texts: list[str]
) -> list[str]:

    encoded = inference_tokenizer(
        input_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(DEVICE)

    generated_ids = inference_model.generate(
        **encoded,
        max_new_tokens=GENERATION_MAX_NEW_TOKENS,
        num_beams=GENERATION_NUM_BEAMS,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

    return inference_tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )


In [40]:
def run_rag_inference(
    retriever_name: str,
    split_name: str,
    top_k: int = RAG_TOP_K,
    batch_size: int = GENERATION_BATCH_SIZE,
) -> Path:

    retrieval_path = (
        RETRIEVAL_DIR
        / f"{retriever_name}_{split_name}_top{TOP_K_EXPORTED}.jsonl"
    )

    if not retrieval_path.exists():
        raise FileNotFoundError(
            f"Nedostaje retrieval fajl: {retrieval_path}"
        )

    retrieval_records = load_jsonl(
        retrieval_path
    )

    outputs = []

    for start in range(
        0,
        len(retrieval_records),
        batch_size
    ):
        batch_records = retrieval_records[
            start:start + batch_size
        ]

        batch_inputs = [
            build_rag_input(
                record,
                top_k=top_k
            )
            for record in batch_records
        ]

        generated_answers = generate_batch(
            batch_inputs
        )

        for record, prediction in zip(
            batch_records,
            generated_answers
        ):
            question_id = record.get(
                "question_id",
                record.get("id")
            )

            outputs.append({
                "question_id": question_id,
                "question": record.get("question", ""),
                "gold_answer": record.get(
                    "answer",
                    record.get("processed_answer", "")
                ),
                "source_pages": record.get("source_pages", []),
                "retriever": retriever_name,
                "generator": EXPERIMENT_NAME,
                "rag_top_k": top_k,
                "generated_answer": prediction.strip(),
                "retrieved_chunks": record.get(
                    "retrieved_chunks",
                    []
                )[:top_k],
            })

    output_path = (
        GENERATION_DIR
        / f"{retriever_name}_{split_name}_top{top_k}_predictions.jsonl"
    )

    save_jsonl(
        outputs,
        output_path
    )

    print(
        f"{retriever_name}: "
        f"{len(outputs)} predictions -> {output_path}"
    )

    return output_path


## Validation inference for all 3 retrievers


In [41]:
RETRIEVERS = (
    "tfidf",
    "bm25",
    "dense",
)

validation_prediction_paths = {}

for retriever_name in RETRIEVERS:
    expected_path = (
        RETRIEVAL_DIR
        / f"{retriever_name}_validation_top{TOP_K_EXPORTED}.jsonl"
    )

    if not expected_path.exists():
        print(
            f"SKIP {retriever_name}: "
            f"nedostaje {expected_path}"
        )
        continue

    validation_prediction_paths[
        retriever_name
    ] = run_rag_inference(
        retriever_name=retriever_name,
        split_name="validation",
        top_k=RAG_TOP_K,
    )

with (ARTIFACTS_DIR / "validation_prediction_files.json").open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        {
            name: str(path)
            for name, path
            in validation_prediction_paths.items()
        },
        file,
        ensure_ascii=False,
        indent=2
    )


tfidf: 21 predictions -> /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/generation/tfidf_validation_top5_predictions.jsonl
bm25: 21 predictions -> /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/generation/bm25_validation_top5_predictions.jsonl
dense: 21 predictions -> /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/generation/dense_validation_top5_predictions.jsonl


## Final test je locked

`RUN_FINAL_TEST = True` when generator is finalized

In [45]:
RUN_FINAL_TEST = True

if RUN_FINAL_TEST:
    test_prediction_paths = {}

    for retriever_name in RETRIEVERS:
        expected_path = (
            RETRIEVAL_DIR
            / f"{retriever_name}_test_top{TOP_K_EXPORTED}.jsonl"
        )

        if not expected_path.exists():
            print(
                f"SKIP {retriever_name}: "
                f"nedostaje {expected_path}"
            )
            continue

        test_prediction_paths[
            retriever_name
        ] = run_rag_inference(
            retriever_name=retriever_name,
            split_name="test",
            top_k=RAG_TOP_K,
        )

    with (ARTIFACTS_DIR / "test_prediction_files.json").open(
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            {
                name: str(path)
                for name, path
                in test_prediction_paths.items()
            },
            file,
            ensure_ascii=False,
            indent=2
        )


tfidf: 22 predictions -> /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/generation/tfidf_test_top5_predictions.jsonl
bm25: 22 predictions -> /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/generation/bm25_test_top5_predictions.jsonl
dense: 22 predictions -> /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/generation/dense_test_top5_predictions.jsonl


## RAG manifesti

Svaka RAG konfiguracija se čuva kao JSON koji vezuje retriever, generator, top-k i output fajl.


In [46]:
GENERATOR_MODEL_DIR = BEST_MODEL_DIR

if not (
    (GENERATOR_MODEL_DIR / "model.safetensors").exists()
    or
    (GENERATOR_MODEL_DIR / "pytorch_model.bin").exists()
):
    raise FileNotFoundError(
        f"Generator weights nisu pronađeni u {GENERATOR_MODEL_DIR}"
    )

for retriever_name in RETRIEVERS:
    retrieval_path = (
        RETRIEVAL_DIR
        / f"{retriever_name}_validation_top{TOP_K_EXPORTED}.jsonl"
    )

    prediction_path = (
        GENERATION_DIR
        / f"{retriever_name}_validation_top{RAG_TOP_K}_predictions.jsonl"
    )

    rag_config = {
        "rag_name": f"{retriever_name}_{EXPERIMENT_NAME}",
        "retriever": retriever_name,
        "retrieval_input": {
            "file": str(retrieval_path),
            "exported_top_k": TOP_K_EXPORTED,
            "rag_top_k": RAG_TOP_K,
        },
        "generator": {
            "experiment": EXPERIMENT_NAME,
            "base_model": MODEL_NAME,
            "saved_model": str(GENERATOR_MODEL_DIR),
            "best_checkpoint": trainer.state.best_model_checkpoint,
            "max_input_length": MAX_INPUT_LENGTH,
            "max_target_length": MAX_TARGET_LENGTH,
            "training_context": "gold_token_budgeted",
        },
        "generation": {
            "num_beams": GENERATION_NUM_BEAMS,
            "max_new_tokens": GENERATION_MAX_NEW_TOKENS,
        },
        "validation_output": (
            str(prediction_path)
            if prediction_path.exists()
            else None
        ),
    }

    output_path = (
        RAG_CONFIG_DIR
        / f"{retriever_name}_{EXPERIMENT_NAME}.json"
    )

    with output_path.open(
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            rag_config,
            file,
            ensure_ascii=False,
            indent=2
        )

    print("Saved:", output_path)


Saved: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/rag_configs/tfidf_mt0_small_gold_v2.json
Saved: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/rag_configs/bm25_mt0_small_gold_v2.json
Saved: /content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold_v2/rag_configs/dense_mt0_small_gold_v2.json


# Sačuvani outputi

Sve se čuva u:

`/content/drive/MyDrive/Student-QA/artifacts/seq2seq/mt0_small_gold/`

Najvažnije:
- `best_model/`
- `checkpoints/`
- `experiment_config.json`
- `training_history.csv`
- `validation_loss_metrics.json`
- `gold_validation_sample_predictions.jsonl`
- `generation/`
- `rag_configs/`
